In [16]:
import pandas as pd
import re

# Step 1: Load Dataset
try:
    df = pd.read_csv('/imdb_clean.csv')
    print("Dataset loaded successfully.")
except FileNotFoundError:
    df = pd.read_csv('imdb_clean.csv')
    print("Dataset loaded successfully.")

# Step 2: Prepare Search Content & Clean All Columns
df['title'] = df['title'].fillna('Unknown')
df['director'] = df['director'].fillna('Unknown')
df['release_year'] = df['release_year'].fillna('N/A')
df['runtime'] = df['runtime'].fillna('N/A')
df['genre'] = df['genre'].fillna('Unknown')
df['rating'] = df['rating'].fillna('N/A')
df['metascore'] = df['metascore'].fillna('N/A')
df['gross(M)'] = df['gross(M)'].fillna('N/A')

# Combine all searchable fields into a single search content column
df['search_content'] = (
    df['title'] + " " +
    df['director'] + " " +
    df['genre'] + " " +
    df['release_year'].astype(str) + " " +
    df['runtime'].astype(str) + " " +
    df['rating'].astype(str)
)

# Step 3: Preprocessing Function
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s.]', '', text) # Keeping '.' to support decimal values like ratings (e.g., 9.3)
    return text.split()

# Step 4: Build Inverted Index
inverted_index = {}
for movie_id, row in df.iterrows():
    tokens = preprocess_text(row['search_content'])
    for token in set(tokens):
        if token not in inverted_index:
            inverted_index[token] = []
        inverted_index[token].append(movie_id)

# Step 5: Boolean Search Function
def search_movies(query, mode='AND'):
    query_tokens = preprocess_text(query)
    if not query_tokens:
        return []

    first_token = query_tokens[0]
    result_set = set(inverted_index.get(first_token, []))

    for token in query_tokens[1:]:
        current_movie_ids = set(inverted_index.get(token, []))
        if mode == 'AND':
            result_set = result_set.intersection(current_movie_ids)
        else:
            result_set = result_set.union(current_movie_ids)

    return list(result_set)

# Step 6: Ranking and Display
def ranked_search(query, mode='AND', is_multi_word=False):
    matched_movie_ids = search_movies(query, mode=mode)
    query_tokens = preprocess_text(query)

    ranked_results = []
    for movie_id in matched_movie_ids:
        movie_text = df.iloc[movie_id]['search_content'].lower()
        score = sum(movie_text.count(token) for token in query_tokens)
        ranked_results.append((movie_id, score))

    ranked_results.sort(key=lambda x: x[1], reverse=True)

    # Clean visual format: displays mode only for multi-word queries
    if is_multi_word:
        print(f"\nResults for: '{query}' (Mode: {mode})")
    else:
        print(f"\nResults for: '{query}'")

    print("-" * 50)

    if not ranked_results:
        print("No results found.")
        return

    for rank, (movie_id, score) in enumerate(ranked_results[:5], 1):
        row = df.iloc[movie_id]

        print(f"{rank}. {row['title']} (Search Score: {score})")
        print(f"   | Release Year : {row['release_year']}")
        print(f"   | Director     : {row['director']}")
        print(f"   | Genre        : {row['genre']}")
        print(f"   | Runtime      : {row['runtime']} mins")
        print(f"   | IMDb Rating  : {row['rating']} / 10")
        print(f"   | Metascore    : {row['metascore']} / 100")
        print(f"   | Gross Profit : ${row['gross(M)']} Million")
        print("-" * 50)

# --- Smart Interactive User Input ---
print("\n=== Movie Search Engine Is Ready ===")
user_query = input("Enter your search keywords: ")

tokens = preprocess_text(user_query)
multi_word_flag = False

# Prompt for AND/OR mode only if the query contains multiple keywords
if len(tokens) > 1:
    multi_word_flag = True
    user_mode = input("Enter search mode (AND / OR): ").strip().upper()
    if user_mode not in ['AND', 'OR']:
        user_mode = 'AND'
else:
    user_mode = 'AND'

# Run the search architecture
ranked_search(user_query, mode=user_mode, is_multi_word=multi_word_flag)

Dataset loaded successfully.

=== Movie Search Engine Is Ready ===
Enter your search keywords: Nolan 1994
Enter search mode (AND / OR): OR

Results for: 'Nolan 1994' (Mode: OR)
--------------------------------------------------
1. The Shawshank Redemption (Search Score: 1)
   | Release Year : 1994
   | Director     : Frank Darabont
   | Genre        : Drama
   | Runtime      : 142 mins
   | IMDb Rating  : 9.3 / 10
   | Metascore    : 82 / 100
   | Gross Profit : $28.34 Million
--------------------------------------------------
2. The Dark Knight (Search Score: 1)
   | Release Year : 2008
   | Director     : Christopher Nolan
   | Genre        : Action
   | Runtime      : 152 mins
   | IMDb Rating  : 9.0 / 10
   | Metascore    : 84 / 100
   | Gross Profit : $534.86 Million
--------------------------------------------------
3. The Dark Knight (Search Score: 1)
   | Release Year : 2008
   | Director     : Christopher Nolan
   | Genre        : Crime
   | Runtime      : 152 mins
   | IMDb R